# 시뮬레이션 -> CSV -> 분석 데모 (E 파트)

E 파트 (경로계획 알고리즘 설계 및 시뮬레이션 구현) / 담당: 백재은

목적
- B0 교체 작업 시뮬레이션을 직접 돌리고, 결과 CSV가 어떻게 만들어지는지 순서대로 보여주기 위한 노트북
- 부품별 선량이 "위치 계산 -> 선량률 조회 -> 시간 적분"으로 나온다는 걸 숫자로 확인하는 것이 핵심

실행 방법
- VS Code에서 이 파일 열기 -> 오른쪽 위 커널 선택에서 conda 환경 `eph` 선택
- 처음 실행할 때 `ipykernel` 설치하라는 창이 뜨면 설치 (한 번만)
- 위에서부터 셀 하나씩 실행 (`Shift + Enter`)
- 전체 소요 약 30 s~1분 (2절 계획 약 10 s, 6절 Xe 베드 계산 약 15 s)

순서
1. 장면 확인
2. 시뮬레이션 실행 (B0)
3. 평가 + CSV 저장
4. 메커니즘 직접 확인 (손계산)
5. 그림
6. Xe 베드 확장 (선택)
7. 정리

참고
- 선량은 가상 선원 기준이라 절대값 X, 부품/단계/알고리즘 사이 상대 비교로만 읽음
- 결과는 `outputs/demo/`에 저장. 기존 `outputs/b0/`는 건드리지 않음

In [ ]:
# 0. 준비: 저장소 경로 잡기 + 패키지 확인
import sys, time, json, csv
from pathlib import Path
import numpy as np
from IPython.display import Image, HTML, display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from datraj import OUTPUTS, config, viz_style, viz
from datraj.scene import Scene
from datraj.task import solve_chain, build_trajectory
from datraj.planning import make_rrt_planner
from datraj.components import load as load_components
from datraj.dose_eval import build_fields, ComponentDoseEvaluator
from datraj.results import evaluate, save

viz_style.apply()
try:
    import toppra  # noqa: F401
    print("toppra 있음 -> 계획 구간 시간 배정에 토크 제약 적용")
except ImportError:
    print("toppra 없음 -> 대체 방식(quintic). 토크 결과 신뢰 X (CLAUDE.md 함정 참고)")

OUT = OUTPUTS / "demo"
OUT.mkdir(parents=True, exist_ok=True)
print("저장소:", ROOT)
print("결과 폴더:", OUT)


def show_table(rows, header, max_rows=None):
    # CSV 일부를 표로 보여주는 작은 함수 (pandas 없이)
    rows = rows if max_rows is None else rows[:max_rows]
    th = "".join(f"<th>{h}</th>" for h in header)
    tr = "".join("<tr>" + "".join(f"<td>{v}</td>" for v in r) + "</tr>" for r in rows)
    display(HTML(f"<table><thead><tr>{th}</tr></thead><tbody>{tr}</tbody></table>"))

## 1. 장면 확인

장면 구성 (`config/hotcell_layout.json`)
- 핫셀 내부 2.0 × 2.0 × 2.0 m, 로봇 KUKA iiwa 14 (7축)
- Bed A: 교체 대상 / Bed B: 교체 중 운전
- 배관(Bed A, Bed B 수직관, 헤더, 출구관)도 약한 선원으로 둠

선원 (`config/virtual_sources.json`)
- 가상 선원. 세기 = 그 선원 전체를 1 m 거리 점선원으로 봤을 때의 선량률 [Gy/h]
- 베드는 축을 따라 균일한 선선원으로 근사

In [ ]:
scene = Scene()
layout = config.layout()
vs = config.virtual_sources()

print("셀 내부:", layout["cell"]["interior_min"], "~", layout["cell"]["interior_max"], "[m]")
print("로봇:", layout["robot"]["model"])
show_table([[s["id"], s["kind"], s["strength"], s.get("note", "")] for s in vs["sources"]],
           ["선원", "형태", "세기 [Gy/h @ 1 m]", "메모"])

fig = ROOT / "docs" / "figures" / "scene_front_left.png"
if fig.exists():
    display(Image(filename=str(fig), width=520))

## 2. 시뮬레이션 실행 (B0)

교체 작업 단계
- S1 접근 / S2 커플링 해제·파지 (체류 60 s) / S3 반출 / S4 신규 베드 장착 이동 / S5 커플링 체결 (체류 60 s) / S6 복귀

B0 알고리즘 (선량을 보지 않는 시간 최소 기준선)
- 웨이포인트 IK로 각 단계 시작/끝 자세 계산
- 이동 구간: RRT-Connect로 충돌 없는 경로 -> 경로 단축 -> TOPP-RA로 시간 배정 (경사 포락선 토크 제약 포함)
- 베드 근처 직선 구간: 5 cm/s 등속, 체류 구간: 자세 고정

아래 셀 실행하면 구간별로 한 줄씩 출력됨
- `[toppra]`: 시간 최소 배정 / `[fine]`: 근접 직선 / `[dwell]`: 체류

In [ ]:
t0 = time.time()
task = config.task_config()
solved = solve_chain(scene, task)
segs = build_trajectory(scene, solved, make_rrt_planner(task), task)
print(f"\n계획 완료 {time.time() - t0:.1f} s, 구간 {len(segs)}개, "
      f"총 작업시간 {sum(s.traj.duration for s in segs):.1f} s")

## 3. 평가 + CSV 저장

평가 내용
- 0.01 s 간격으로 관절각 q -> 부품 10개(C01~C10)와 TCP(P0) 위치 -> 선량률 -> 누적선량
- 같은 시점에서 경사 포락선 7조건 토크 사용률도 계산

저장되는 파일 (`outputs/demo/`)
- `traj_demo.csv`: 시점마다 한 줄. 시간, 단계, 관절각/속도/가속도, 토크, 토크 사용률
- `dose_by_component_demo.csv`: 부품별 단계별 선량, 합계, 최대 선량률
- `summary_demo.json`: 요약 수치

In [ ]:
comps, settings = load_components()
fields = build_fields("model")
t0 = time.time()
E = evaluate(segs, scene, fields, comps, settings, label="B0 demo")
save(E, segs, comps, OUT, "demo")
print(f"평가 {time.time() - t0:.1f} s")
for f in sorted(OUT.glob("*_demo.*")):
    print(" ", f.name)

In [ ]:
# traj_demo.csv 앞부분 (열이 많아서 일부만)
with open(OUT / "traj_demo.csv", encoding="utf-8") as f:
    rows = list(csv.reader(f))
header, body = rows[0], rows[1:]
keep = ["t", "stage", "kind", "q_A1", "q_A2", "q_A4", "q_A6", "tau_A2", "tau_A6", "util_envelope_max"]
idx = [header.index(k) for k in keep]
print(f"행 수 {len(body)} (0.01 s 간격), 열 수 {len(header)}")
print("토크(tau, util)는 이동 구간에서 5개 샘플마다 계산해서 나머지 행은 빈칸. 아래는 토크가 있는 행만 골라 일부 표시")
ti = header.index("tau_A2")
with_tau = [r for r in body if r[ti] != ""]
show_table([[r[i] for i in idx] for r in with_tau[::120]], keep)

In [ ]:
# dose_by_component_demo.csv 전체
with open(OUT / "dose_by_component_demo.csv", encoding="utf-8") as f:
    rows = list(csv.reader(f))
header, body = rows[0], rows[1:]
cols = ["id", "name", "D_S1_Gy", "D_S2_Gy", "D_S3_Gy", "D_S4_Gy", "D_S5_Gy", "D_S6_Gy", "D_total_Gy"]
idx = [header.index(c) for c in cols]
show_table([[r[idx[0]], r[idx[1]]] + [f"{float(r[i]) * 1000:.1f}" for i in idx[2:]] for r in body],
           ["부품", "이름", "S1", "S2", "S3", "S4", "S5", "S6", "합계 [mGy]"])

## 4. 메커니즘 직접 확인 (손계산)

CSV의 선량이 어떻게 나오는지 직접 계산해서 맞춰 봄

계산 순서
1) 관절각 q -> 순기구학 -> 부품 위치 p_k (링크 좌표계에 붙인 점을 월드 좌표로)
2) 위치 p_k -> 선량장에서 선량률 Ḋ(p_k) 조회 [Gy/h]
3) 시간 순서대로 Ḋ × Δt 적분 -> 누적선량 D_k = ∫ Ḋ(p_k(q(t))) dt

### 4-1. 한 시점 확인

S2 체류(커플링 해제) 중간 시점 하나를 골라서 C09 손목 카메라와 TCP의 위치·선량률 계산

In [ ]:
ev = ComponentDoseEvaluator(scene.robot, comps)
ids = [c.id for c in comps]
k09, kTCP = ids.index("C09"), ids.index("P0")

# S2 체류 구간 가운데 시점
s2 = [i for i, s in enumerate(E.stage) if s == "S2"]
i = s2[len(s2) // 2]
seg = segs[E.seg_index[i]]
field = fields[seg.step.field]

P, tcp, R = ev.positions(E.q[i][None, :])          # 1) 순기구학
rate = field.rate(P, tcp, R, E.t[i:i + 1])[0]       # 2) 선량률 조회

print(f"시점 t = {E.t[i]:.2f} s, 단계 {E.stage[i]}, 구간 '{seg.step.name}', 선량장 '{field.name}'")
print(f"C09 위치 {np.round(P[0, k09], 3)} m -> 선량률 {rate[k09]:.3f} Gy/h (평가 결과 {E.rates[i, k09]:.3f})")
print(f"TCP 위치 {np.round(P[0, kTCP], 3)} m -> 선량률 {rate[kTCP]:.3f} Gy/h (평가 결과 {E.rates[i, kTCP]:.3f})")

### 4-2. 시간 적분 확인

구간마다 2)까지 반복 -> 사다리꼴 적분 -> 전체 구간 합 = CSV `D_total_Gy`

In [ ]:
D_hand = np.zeros(len(comps))
for sg in segs:
    tr = sg.traj
    P, tcp, R = ev.positions(tr.q)
    r = fields[sg.step.field].rate(P, tcp, R, tr.t)      # (시점 수, 부품 수) Gy/h
    D_hand += np.trapezoid(r, tr.t, axis=0) / 3600.0     # Gy/h × s / 3600 = Gy

with open(OUT / "dose_by_component_demo.csv", encoding="utf-8") as f:
    csv_total = {row["id"]: float(row["D_total_Gy"]) for row in csv.DictReader(f)}

rows = []
for c, d in zip(comps, D_hand):
    rows.append([c.id, c.name, f"{d * 1000:.3f}", f"{csv_total[c.id] * 1000:.3f}",
                 f"{(d / csv_total[c.id] - 1) * 100:+.4f}"])
show_table(rows, ["부품", "이름", "손계산 [mGy]", "CSV [mGy]", "차이 [%]"])

### 4-3. 점 로봇(TCP) 가정과 비교

E 파트 주장: 로봇을 TCP 한 점으로 보면 부품 선량을 대표하지 못함

In [ ]:
real = [k for k, c in enumerate(comps) if not c.reference]
kmax = real[int(np.argmax(E.dose_total[real]))]
print(f"최대 선량 부품: {comps[kmax].id} {comps[kmax].name}")
print(f"TCP / 최대 부품 = {E.dose_total[kTCP] / E.dose_total[kmax]:.2f}배")
print(f"최대 부품 / 최소 부품 = {E.dose_total[real].max() / E.dose_total[real].min():.1f}배")

by_kind = {"plan": 0.0, "line": 0.0, "dwell": 0.0}
for sg, d in zip(segs, E.seg_dose):
    by_kind[sg.step.kind] += float(np.sum(d[real]))
tot = sum(by_kind.values())
print("\n부품 선량 합의 구성")
for k, name in (("dwell", "체류"), ("line", "베드 근처 직선"), ("plan", "경로계획 구간")):
    print(f"  {name}: {by_kind[k] / tot * 100:.0f}%")

## 5. 그림

- `path_demo.png`: 위에서 본 TCP 경로 + 선량장
- `timeline_demo.png`: 시간에 따른 부품별 선량률, 단계 구분
- `dose_components_demo.png`: 부품별 단계별 누적선량

In [ ]:
viz.trajectory_figures(E, segs, comps, scene, fields, OUT, "demo")
for name in ("timeline_demo.png", "dose_components_demo.png", "path_demo.png"):
    display(Image(filename=str(OUT / name), width=760))

### (선택) PyBullet 창에서 재생

노트북 안에서 GUI 창을 띄우면 Mac에서 멈추는 경우가 있어서 기본은 꺼둠
- 발표 시연은 터미널에서 `python scripts/06_run_b0.py --gui --speed 3` 권장 (같은 B0 궤적)
- 노트북에서 해보려면 `RUN_GUI = True`로 바꾸고 실행. 창을 닫으면 끝남

In [ ]:
RUN_GUI = False
if RUN_GUI:
    scene.disconnect()
    viz.replay_gui(E, segs, fields, speed=3.0)

## 6. Xe 베드 확장 (선택)

10/06 회의 결정(Xe 단일 타겟, 이중베드) 반영 계산 (`scripts/11_xe_bed_budget.py`)
- 위 B0 궤적의 부품별 단위 선량 × MARINA 식 Xe 선원 상한 -> 교체 1회 선량
- 교체 주기/냉각/차폐/로봇 하중을 묶어서 가능한 조건 찾기
- 자세한 해석은 `docs/2026-10-11_Xe베드_교체조건_분석.md`

약 15 s 소요

In [ ]:
import subprocess
res = subprocess.run([sys.executable, str(ROOT / "scripts" / "11_xe_bed_budget.py")],
                     cwd=ROOT, capture_output=True, text=True)
keys = ("교체 1회", "흡착제 소요량", "Bed A 선원 세기")
print("\n".join(l for l in res.stdout.splitlines() if l.startswith(keys)) or res.stderr[-2000:])

xe = OUTPUTS / "xe_bed"
with open(xe / "budget_table.csv", encoding="utf-8") as f:
    rows = [r for r in csv.DictReader(f)
            if r["material"] == "W" and r["condition"] == "0 °C, 7 bar" and r["concept"].startswith("C1")
            and r["B_fixed_shield"] == "True" and r["T_on_d"] in ("7", "14")]
show_table([[r["T_on_d"], r["t_c_d"], r["sorbent_kg"], r["N_ex"], r["x_req_mm"], r["module_kg"],
             r["module_kg_yearly_parts"], r["robots_ok_yearly_parts"]] for r in rows],
           ["T_on [일]", "t_c [일]", "흡착제 [kg]", "교체 횟수", "W [mm]", "모듈 [kg]", "부품 매년 교체 시 [kg]", "가능 로봇"])
display(Image(filename=str(xe / "xe_bed_budget.png"), width=900))

## 7. 정리

설명 포인트
1) 시뮬레이션: 관절각 시계열(q)이 결과의 전부. 0.01 s 간격으로 `traj_demo.csv`에 저장됨
2) 분석: q -> 부품 위치 -> 선량률 -> 시간 적분. 손계산(4-2)과 CSV가 일치하는 것으로 확인
3) 결과: 부품마다 선량이 다르고 TCP 한 점으로는 대표 X, 선량 대부분은 체류 구간에서 발생
4) 확장: 같은 단위 선량에 Xe 베드 선원을 곱해서 교체 조건(주기/냉각/차폐/로봇 하중) 계산

주의
- 1~5절 선량은 가상 선원 기준 -> 상대 비교만
- 6절 선량은 MARINA 식 상한 추정 (ε = 1, 자체차폐/빌드업 없음)